# Agente 09 — Multi-Agente con RAG (Pink Panther)

Este notebook implementa un sistema **multi-agente** con `smolagents` donde:

- Un **Manager** (Pink Panther) orquesta 5 subagentes especializados
- Cada subagente tiene su propia tool especializada
- El manager tiene acceso a una **RAG Tool** con el lore del universo
- Se usa `managed_agents` para delegación jerárquica

**Modelo:** Claude Sonnet 4 vía Amazon Bedrock

**Arquitectura:**
```
Pink Panther Manager
├── RAG Tool (lore del universo)
├── Rey Estratega (subagente)
├── Vanidoso Arte (subagente)
├── Hombre de Negocios Quant (subagente)
├── Farolero Energía (subagente)
└── Geógrafo Fantasía (subagente)
```

## 1. Instalación de dependencias

In [17]:
!uv pip install "smolagents[litellm]" sentence-transformers numpy boto3 --quiet

## 2. Configuración del token de Bedrock

El token se lee desde los **Secretos de Google Colab**.

Para configurarlo: Panel izquierdo → 🔑 Secretos → Agregar `AWS_BEARER_TOKEN_BEDROCK` con tu token.

In [18]:
import os
from google.colab import userdata

os.environ["AWS_BEARER_TOKEN_BEDROCK"] = userdata.get("AWS_BEARER_TOKEN_BEDROCK")
os.environ["AWS_REGION_NAME"] = "us-east-1"

## 3. Configuración del modelo LLM

In [19]:
from smolagents import Tool, ToolCallingAgent, LiteLLMModel

model = LiteLLMModel(
    model_id="bedrock/converse/us.anthropic.claude-sonnet-4-20250514-v1:0",
    aws_region_name="us-east-1",
)

## 4. Base de conocimiento (documentos para RAG)

In [20]:
DOCUMENTOS = [
    """Pink Panther es una aventurera elegante que resuelve problemas
con inteligencia, humor, sigilo y creatividad.""",
    """El Rey es experto en estrategia de negocios, liderazgo,
negociación y toma de decisiones.""",
    """El Vanidoso domina arte, estética, símbolos, belleza,
composición visual y narrativa artística.""",
    """El Hombre de Negocios es experto en finanzas cuantitativas,
simulación, riesgo, volatilidad y escenarios futuros.""",
    """El Farolero es experto en energía eléctrica, resiliencia,
infraestructura crítica y continuidad operativa.""",
    """El Geógrafo conoce mapas, territorios, ciudades antiguas,
rutas secretas y mundos de fantasía.""",
    """El reino rosa sufrió un apagón energético después de una
tormenta mágica.""",
    """En una galería de arte antigua se oculta un mapa secreto
que conduce al corazón energético del reino.""",
]

## 5. RAG Tool (búsqueda semántica en el lore)

In [21]:
import numpy as np
from sentence_transformers import SentenceTransformer


class PinkPantherRAGTool(Tool):
    name = "pink_panther_rag"
    description = "Busca información en el lore del universo de Pink Panther."
    inputs = {
        "query": {
            "type": "string",
            "description": "Pregunta o tema a buscar en el lore."
        }
    }
    output_type = "string"

    def __init__(self):
        super().__init__()
        self.encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        self.docs = DOCUMENTOS
        self.embeddings = self.encoder.encode(
            self.docs,
            normalize_embeddings=True
        )

    def forward(self, query: str) -> str:
        query_embedding = self.encoder.encode(
            [query],
            normalize_embeddings=True
        )[0]
        scores = np.dot(self.embeddings, query_embedding)
        top_idx = scores.argsort()[-3:][::-1]
        resultados = [self.docs[i] for i in top_idx]
        return "\n".join(resultados)

## 6. Tools especializadas (una por subagente)

In [22]:
class EstrategiaTool(Tool):
    name = "analizar_estrategia"
    description = "Analiza una situación desde estrategia, liderazgo y negociación."
    inputs = {
        "situacion": {
            "type": "string",
            "description": "Situación estratégica a analizar."
        }
    }
    output_type = "string"

    def forward(self, situacion: str) -> str:
        return f"""Diagnóstico estratégico:
Situación: {situacion}
Recomendación:
1. Identificar el objetivo principal.
2. Separar aliados, adversarios y recursos.
3. Priorizar acciones de bajo riesgo y alto impacto.
4. Proteger la reputación del reino.
5. Actuar con paciencia, como buen soberano."""

In [23]:
class ArteTool(Tool):
    name = "analizar_arte"
    description = "Analiza una escena desde arte, estética y símbolos."
    inputs = {
        "escena": {
            "type": "string",
            "description": "Escena artística o narrativa visual."
        }
    }
    output_type = "string"

    def forward(self, escena: str) -> str:
        return f"""Lectura artística:
Escena: {escena}
Recomendación:
Usar contraste entre rosa, azul profundo y dorado.
El mapa debe aparecer como símbolo de memoria antigua.
La galería debe sentirse como un templo del conocimiento."""

In [24]:
class RiesgoQuantTool(Tool):
    name = "analizar_riesgo_quant"
    description = "Analiza una decisión usando riesgo, simulación y escenarios."
    inputs = {
        "problema": {
            "type": "string",
            "description": "Problema financiero o de riesgo."
        }
    }
    output_type = "string"

    def forward(self, problema: str) -> str:
        return f"""Análisis cuantitativo:
Problema: {problema}
Escenarios:
- Escenario base: recuperación gradual del reino.
- Escenario adverso: el apagón se extiende.
- Escenario extremo: falla el corazón energético.
Recomendación:
Pink Panther debe tomar decisiones robustas,
no decisiones óptimas para un solo escenario."""

In [25]:
class EnergiaTool(Tool):
    name = "analizar_energia"
    description = "Analiza incidentes de energía, resiliencia e infraestructura."
    inputs = {
        "incidente": {
            "type": "string",
            "description": "Incidente energético o de infraestructura."
        }
    }
    output_type = "string"

    def forward(self, incidente: str) -> str:
        return f"""Diagnóstico energético:
Incidente: {incidente}
Plan:
1. Restaurar primero servicios críticos.
2. Revisar el corazón energético del reino.
3. Activar fuentes redundantes.
4. Monitorear estabilidad.
5. Evitar reconexión total sin pruebas."""

In [26]:
class MapaFantasiaTool(Tool):
    name = "crear_mapa_fantasia"
    description = "Diseña mapas, regiones y rutas de un mundo fantástico."
    inputs = {
        "mundo": {
            "type": "string",
            "description": "Descripción del mundo o territorio."
        }
    }
    output_type = "string"

    def forward(self, mundo: str) -> str:
        return f"""Mapa fantástico:
Mundo: {mundo}
Regiones:
1. Palacio del Rey.
2. Galería del Vanidoso.
3. Mercado del Hombre de Negocios.
4. Torre del Farolero.
5. Bosque Rosa del Geógrafo.
Ruta sugerida:
Galería antigua -> Torre del Farolero -> Corazón energético."""

## 7. Creación de subagentes especializados

In [27]:
rey_agent = ToolCallingAgent(
    name="rey_estratega",
    description="""Agente experto en estrategia de negocios, liderazgo,
negociación y toma de decisiones para ayudar a Pink Panther.""",
    tools=[EstrategiaTool()],
    model=model,
    max_steps=3,
)

vanidoso_agent = ToolCallingAgent(
    name="vanidoso_arte",
    description="""Agente experto en arte, estética, símbolos y narrativa visual.
Ayuda a interpretar galerías, mapas y escenas.""",
    tools=[ArteTool()],
    model=model,
    max_steps=3,
)

hombre_negocios_agent = ToolCallingAgent(
    name="hombre_negocios_quant",
    description="""Agente experto en finanzas cuantitativas, simulación,
riesgo y escenarios futuros.""",
    tools=[RiesgoQuantTool()],
    model=model,
    max_steps=3,
)

farolero_agent = ToolCallingAgent(
    name="farolero_energia",
    description="""Agente experto en energía eléctrica, resiliencia,
infraestructura crítica y continuidad operativa.""",
    tools=[EnergiaTool()],
    model=model,
    max_steps=3,
)

geografo_agent = ToolCallingAgent(
    name="geografo_fantasia",
    description="""Agente experto en mapas, territorios, rutas secretas
y mundos de fantasía.""",
    tools=[MapaFantasiaTool()],
    model=model,
    max_steps=3,
)

print("✅ 5 subagentes creados exitosamente")

✅ 5 subagentes creados exitosamente


## 8. Agente Manager — Pink Panther

El manager tiene acceso al RAG y delega trabajo a los 5 subagentes.

In [28]:
pink_panther_manager = ToolCallingAgent(
    name="pink_panther_manager",
    description="""Pink Panther coordina especialistas para resolver aventuras
en un mundo fantástico. Primero consulta el lore con RAG,
después delega trabajo a subagentes especializados.""",
    tools=[PinkPantherRAGTool()],
    managed_agents=[
        rey_agent,
        vanidoso_agent,
        hombre_negocios_agent,
        farolero_agent,
        geografo_agent,
    ],
    model=model,
    max_steps=10,
)

print("✅ Manager Pink Panther listo con 5 managed_agents + RAG")

✅ Manager Pink Panther listo con 5 managed_agents + RAG


## 9. Ejecución — La aventura de Pink Panther

In [29]:
prompt = """Pink Panther llega a un reino fantástico que sufrió un apagón energético.
Hay rumores de que en una galería de arte antigua está escondido un mapa
que conduce al corazón energético del reino.

Primero consulta el lore disponible.
Después pide ayuda a los subagentes necesarios:
- Rey Estratega
- Vanidoso Arte
- Hombre de Negocios Quant
- Farolero Energía
- Geógrafo Fantasía

Finalmente entrega:
1. Diagnóstico de la situación.
2. Plan de aventura.
3. Riesgos principales.
4. Ruta por el mundo fantástico.
5. Final narrativo elegante al estilo Pink Panther."""

respuesta = pink_panther_manager.run(prompt)

╭──────────────────────────────────────── New run - pink_panther_manager ─────────────────────────────────────────╮
│                                                                                                                 │
│ Pink Panther llega a un reino fantástico que sufrió un apagón energético.                                       │
│ Hay rumores de que en una galería de arte antigua está escondido un mapa                                        │
│ que conduce al corazón energético del reino.                                                                    │
│                                                                                                                 │
│ Primero consulta el lore disponible.                                                                            │
│ Después pide ayuda a los subagentes necesarios:                                                                 │
│ - Rey Estratega                                                                                                 │
│ - Vanidoso Arte                                                                                                 │
│ - Hombre de Negocios Quant                                                                                      │
│ - Farolero Energía                                                                                              │
│ - Geógrafo Fantasía                                                                                             │
│                                                                                                                 │
│ Finalmente entrega:                                                                                             │
│ 1. Diagnóstico de la situación.                                                                                 │
│ 2. Plan de aventura.                                                                                            │
│ 3. Riesgos principales.                                                                                         │
│ 4. Ruta por el mundo fantástico.                                                                                │
│ 5. Final narrativo elegante al estilo Pink Panther.                                                             │
│                                                                                                                 │
╰─ LiteLLMModel - bedrock/converse/us.anthropic.claude-sonnet-4-20250514-v1:0 ────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'pink_panther_rag' with arguments: {'query': 'Pink Panther aventuras reino fantástico apagón      │
│ energético galería arte mapa corazón energético'}                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Pink Panther es una aventurera elegante que resuelve problemas
con inteligencia, humor, sigilo y creatividad.
En una galería de arte antigua se oculta un mapa secreto
que conduce al corazón energético del reino.
El reino rosa sufrió un apagón energético después de una
tormenta mágica.

[Step 1: Duration 5.00 seconds| Input tokens: 3,195 | Output tokens: 66]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': '**INFORME COMPLETO: MISIÓN REINO ROSA**\n\n## 1.       │
│ DIAGNÓSTICO DE LA SITUACIÓN\n\n**Estado Actual del Reino Rosa:**\n- **Crisis Energética:** Apagón total causado │
│ por tormenta mágica que disrumpió el corazón energético\n- **Impacto Sistémico:** Falla en infraestructura      │
│ crítica, servicios esenciales y defensas del reino\n- **Condiciones Ambientales:** Nieblas rosadas, terreno     │
│ post-tormenta, criaturas sombra emergentes\n- **Oportunidades Tácticas:** Oscuridad energética favorece         │
│ operaciones sigilosas, rutas alternativas disponibles\n\n**Recurso Clave Identificado:**\n- Mapa secreto oculto │
│ en la Galería del Vanidoso (galería de arte antigua)\n- Contiene la ubicación exacta del corazón energético del │
│ reino\n\n## 2. PLAN DE AVENTURA\n\n**FASE 1 - INFILTRACIÓN ARTÍSTICA (0-6 horas)**\n- Acceso elegante a la      │
│ Galería del Vanidoso usando túneles de contrabandistas\n- Análisis sistemático de obras con técnicas no         │
│ destructivas (luz UV, resonancia térmica)\n- Búsqueda de símbolos rosa-dorado y códigos cromáticos en marcos    │
│ secretos\n\n**FASE 2 - DECODIFICACIÓN CULTURAL (6-12 horas)**\n- Interpretación de símbolos: cristales rosados, │
│ espirales doradas, laberintos energéticos\n- Activación del mapa mediante secuencias gestuales o ronroneo       │
│ específico\n- Documentación discreta de la ruta al corazón energético\n\n**FASE 3 - NAVEGACIÓN ESTRATÉGICA      │
│ (12-18 horas)**\n- Ruta óptima: Galería → Torre del Farolero → Corazón Energético\n- Uso de nieblas rosadas     │
│ como cobertura, evitando criaturas sombra\n- Construcción de alianzas con habitantes locales (artistas,         │
│ bibliotecarios)\n\n**FASE 4 - RESTAURACIÓN ENERGÉTICA (18-72 horas)**\n- Diagnóstico técnico del corazón        │
│ energético: integridad estructural y energía residual\n- Reactivación gradual: 25% → 50% → 75% → 100% de        │
│ capacidad\n- Reconexión progresiva de servicios críticos → sectores completos\n\n## 3. RIESGOS                  │
│ PRINCIPALES\n\n**Riesgos Inmediatos:**\n- **Trampas en la galería:** Marcos con mecanismos de seguridad, falsas │
│ pistas\n- **Criaturas sombra:** Entidades hostiles surgidas del apagón\n- **Inestabilidad climática:**          │
│ Tormentas residuales, derrumbes en montañas de cuarzo\n\n**Riesgos Operacionales:**\n- **Falla del corazón      │
│ energético:** Daño permanente requiriendo soluciones alternativas\n- **Detección enemiga:** Vigilancia de la    │
│ galería o el corazón energético\n- **Cascada de fallos:** Reconexión prematura causando daños                   │
│ sistémicos\n\n**Contingencias Desarrolladas:**\n- Múltiples rutas de escape desde cada ubicación\n- Alianzas    │
│ locales como red de apoyo\n- Estrategias alternativas de restauración energética\n\n## 4. RUTA POR EL MUNDO     │
│ FANTÁSTICO\n\n**MAPA ESTRATÉGICO DEL REINO ROSA:**\n\n```\n🏰 Palacio del Rey (Norte) - Centro político,        │
│ evitar\n     ↓\n🎨 Galería del Vanidoso (Centro-Este) - OBJETIVO PRIMARIO\n     ↓ (Túneles subterráneos)\n🗼    │
│ Torre del Farolero (Sur-Este) - Punto de navegación\n     ↓ (Senderos de montaña)\n💎 Corazón Energético        │
│ (Centro-Sur) - OBJETIVO FINAL\n```\n\n**Características del Terreno:**\n- **Montañas de Cuarzo Rosa:**          │
│ Cobertura elevada, rutas alternativas\n- **Bosques Sin Brillo:** Camuflaje natural, visibilidad reducida\n-     │
│ **Valles con Jardines Marchitos:** Terreno difícil pero cobertura\n- **Túneles de Contrabandistas:** Red        │
│ subterránea para movimiento sigiloso\n\n**Puntos de Interés Táctico:**\n- **Mercado del Hombre de Negocios:**   │
│ Suministros y camuflaje\n- **Bosque Rosa del Geógrafo:** Rutas de escape hacia la frontera\n\n## 5. FINAL       │
│ NARRATIVO ELEGANTE AL ESTILO PINK PANTHER\n\n*Con la graci

Observations: **INFORME COMPLETO: MISIÓN REINO ROSA**

## 1. DIAGNÓSTICO DE LA SITUACIÓN

**Estado Actual del Reino Rosa:**
- **Crisis Energética:** Apagón total causado por tormenta mágica que disrumpió el corazón energético
- **Impacto Sistémico:** Falla en infraestructura crítica, servicios esenciales y defensas del reino
- **Condiciones Ambientales:** Nieblas rosadas, terreno post-tormenta, criaturas sombra emergentes
- **Oportunidades Tácticas:** Oscuridad energética favorece operaciones sigilosas, rutas alternativas disponibles

**Recurso Clave Identificado:**
- Mapa secreto oculto en la Galería del Vanidoso (galería de arte antigua)
- Contiene la ubicación exacta del corazón energético del reino

## 2. PLAN DE AVENTURA

**FASE 1 - INFILTRACIÓN ARTÍSTICA (0-6 horas)**
- Acceso elegante a la Galería del Vanidoso usando túneles de contrabandistas
- Análisis sistemático de obras con técnicas no destructivas (luz UV, resonancia térmica)
- Búsqueda de símbolos rosa-dorado y códigos cromáticos en marcos secretos

**FASE 2 - DECODIFICACIÓN CULTURAL (6-12 horas)**
- Interpretación de símbolos: cristales rosados, espirales doradas, laberintos energéticos
- Activación del mapa mediante secuencias gestuales o ronroneo específico
- Documentación discreta de la ruta al corazón energético

**FASE 3 - NAVEGACIÓN ESTRATÉGICA (12-18 horas)**
- Ruta óptima: Galería → Torre del Farolero → Corazón Energético
- Uso de nieblas rosadas como cobertura, evitando criaturas sombra
- Construcción de alianzas con habitantes locales (artistas, bibliotecarios)

**FASE 4 - RESTAURACIÓN ENERGÉTICA (18-72 horas)**
- Diagnóstico técnico del corazón energético: integridad estructural y energía residual
- Reactivación gradual: 25% → 50% → 75% → 100% de capacidad
- Reconexión progresiva de servicios críticos → sectores completos

## 3. RIESGOS PRINCIPALES

**Riesgos Inmediatos:**
- **Trampas en la galería:** Marcos con mecanismos de seguridad, falsas pistas
- **Criaturas sombra:** Entidades hostiles surgidas del apagón
- **Inestabilidad climática:** Tormentas residuales, derrumbes en montañas de cuarzo

**Riesgos Operacionales:**
- **Falla del corazón energético:** Daño permanente requiriendo soluciones alternativas
- **Detección enemiga:** Vigilancia de la galería o el corazón energético
- **Cascada de fallos:** Reconexión prematura causando daños sistémicos

**Contingencias Desarrolladas:**
- Múltiples rutas de escape desde cada ubicación
- Alianzas locales como red de apoyo
- Estrategias alternativas de restauración energética

## 4. RUTA POR EL MUNDO FANTÁSTICO

**MAPA ESTRATÉGICO DEL REINO ROSA:**

```
🏰 Palacio del Rey (Norte) - Centro político, evitar
     ↓
🎨 Galería del Vanidoso (Centro-Este) - OBJETIVO PRIMARIO
     ↓ (Túneles subterráneos)
🗼 Torre del Farolero (Sur-Este) - Punto de navegación
     ↓ (Senderos de montaña)
💎 Corazón Energético (Centro-Sur) - OBJETIVO FINAL
```

**Características del Terreno:**
- **Montañas de Cuarzo Rosa:** Cobertura elevada, rutas alternativas
- **Bosques Sin Brillo:** Camuflaje natural, visibilidad reducida
- **Valles con Jardines Marchitos:** Terreno difícil pero cobertura
- **Túneles de Contrabandistas:** Red subterránea para movimiento sigiloso

**Puntos de Interés Táctico:**
- **Mercado del Hombre de Negocios:** Suministros y camuflaje
- **Bosque Rosa del Geógrafo:** Rutas de escape hacia la frontera

## 5. FINAL NARRATIVO ELEGANTE AL ESTILO PINK PANTHER

*Con la gracia característica de una auténtica dama ladrona, Pink Panther emerge de las sombras rosadas de la 
galería, sosteniendo delicadamente entre sus garras el mapa ancestral que brilla con un suave resplandor dorado. 
Sus ojos verdes centellean con la satisfacción del trabajo bien hecho.*

*"Mes amis,"* susurra con su elegante acento, deslizándose por los túneles secretos como una sombra de terciopelo, 
*"la belleza del arte siempre revela sus secretos a quienes saben apreciarla con... finesse."*

*Siguiendo las pistas cromáticas del mapa—cristales ros

Final answer: **INFORME COMPLETO: MISIÓN REINO ROSA**

## 1. DIAGNÓSTICO DE LA SITUACIÓN

**Estado Actual del Reino Rosa:**
- **Crisis Energética:** Apagón total causado por tormenta mágica que disrumpió el corazón energético
- **Impacto Sistémico:** Falla en infraestructura crítica, servicios esenciales y defensas del reino
- **Condiciones Ambientales:** Nieblas rosadas, terreno post-tormenta, criaturas sombra emergentes
- **Oportunidades Tácticas:** Oscuridad energética favorece operaciones sigilosas, rutas alternativas disponibles

**Recurso Clave Identificado:**
- Mapa secreto oculto en la Galería del Vanidoso (galería de arte antigua)
- Contiene la ubicación exacta del corazón energético del reino

## 2. PLAN DE AVENTURA

**FASE 1 - INFILTRACIÓN ARTÍSTICA (0-6 horas)**
- Acceso elegante a la Galería del Vanidoso usando túneles de contrabandistas
- Análisis sistemático de obras con técnicas no destructivas (luz UV, resonancia térmica)
- Búsqueda de símbolos rosa-dorado y códigos cromáticos en marcos secretos

**FASE 2 - DECODIFICACIÓN CULTURAL (6-12 horas)**
- Interpretación de símbolos: cristales rosados, espirales doradas, laberintos energéticos
- Activación del mapa mediante secuencias gestuales o ronroneo específico
- Documentación discreta de la ruta al corazón energético

**FASE 3 - NAVEGACIÓN ESTRATÉGICA (12-18 horas)**
- Ruta óptima: Galería → Torre del Farolero → Corazón Energético
- Uso de nieblas rosadas como cobertura, evitando criaturas sombra
- Construcción de alianzas con habitantes locales (artistas, bibliotecarios)

**FASE 4 - RESTAURACIÓN ENERGÉTICA (18-72 horas)**
- Diagnóstico técnico del corazón energético: integridad estructural y energía residual
- Reactivación gradual: 25% → 50% → 75% → 100% de capacidad
- Reconexión progresiva de servicios críticos → sectores completos

## 3. RIESGOS PRINCIPALES

**Riesgos Inmediatos:**
- **Trampas en la galería:** Marcos con mecanismos de seguridad, falsas pistas
- **Criaturas sombra:** Entidades hostiles surgidas del apagón
- **Inestabilidad climática:** Tormentas residuales, derrumbes en montañas de cuarzo

**Riesgos Operacionales:**
- **Falla del corazón energético:** Daño permanente requiriendo soluciones alternativas
- **Detección enemiga:** Vigilancia de la galería o el corazón energético
- **Cascada de fallos:** Reconexión prematura causando daños sistémicos

**Contingencias Desarrolladas:**
- Múltiples rutas de escape desde cada ubicación
- Alianzas locales como red de apoyo
- Estrategias alternativas de restauración energética

## 4. RUTA POR EL MUNDO FANTÁSTICO

**MAPA ESTRATÉGICO DEL REINO ROSA:**

```
🏰 Palacio del Rey (Norte) - Centro político, evitar
     ↓
🎨 Galería del Vanidoso (Centro-Este) - OBJETIVO PRIMARIO
     ↓ (Túneles subterráneos)
🗼 Torre del Farolero (Sur-Este) - Punto de navegación
     ↓ (Senderos de montaña)
💎 Corazón Energético (Centro-Sur) - OBJETIVO FINAL
```

**Características del Terreno:**
- **Montañas de Cuarzo Rosa:** Cobertura elevada, rutas alternativas
- **Bosques Sin Brillo:** Camuflaje natural, visibilidad reducida
- **Valles con Jardines Marchitos:** Terreno difícil pero cobertura
- **Túneles de Contrabandistas:** Red subterránea para movimiento sigiloso

**Puntos de Interés Táctico:**
- **Mercado del Hombre de Negocios:** Suministros y camuflaje
- **Bosque Rosa del Geógrafo:** Rutas de escape hacia la frontera

## 5. FINAL NARRATIVO ELEGANTE AL ESTILO PINK PANTHER

*Con la gracia característica de una auténtica dama ladrona, Pink Panther emerge de las sombras rosadas de la 
galería, sosteniendo delicadamente entre sus garras el mapa ancestral que brilla con un suave resplandor dorado. 
Sus ojos verdes centellean con la satisfacción del trabajo bien hecho.*

*"Mes amis,"* susurra con su elegante acento, deslizándose por los túneles secretos como una sombra de terciopelo, 
*"la belleza del arte siempre revela sus secretos a quienes saben apreciarla con... finesse."*

*Siguiendo las pistas cromáticas del mapa—cristales ros

[Step 3: Duration 24.01 seconds| Input tokens: 17,848 | Output tokens: 3,000]

## 10. Resultado final

In [30]:
print("================================================")
print("RESPUESTA FINAL DEL MANAGER PINK PANTHER")
print("================================================")
print()
print(respuesta)

RESPUESTA FINAL DEL MANAGER PINK PANTHER

**INFORME COMPLETO: MISIÓN REINO ROSA**

## 1. DIAGNÓSTICO DE LA SITUACIÓN

**Estado Actual del Reino Rosa:**
- **Crisis Energética:** Apagón total causado por tormenta mágica que disrumpió el corazón energético
- **Impacto Sistémico:** Falla en infraestructura crítica, servicios esenciales y defensas del reino
- **Condiciones Ambientales:** Nieblas rosadas, terreno post-tormenta, criaturas sombra emergentes
- **Oportunidades Tácticas:** Oscuridad energética favorece operaciones sigilosas, rutas alternativas disponibles

**Recurso Clave Identificado:**
- Mapa secreto oculto en la Galería del Vanidoso (galería de arte antigua)
- Contiene la ubicación exacta del corazón energético del reino

## 2. PLAN DE AVENTURA

**FASE 1 - INFILTRACIÓN ARTÍSTICA (0-6 horas)**
- Acceso elegante a la Galería del Vanidoso usando túneles de contrabandistas
- Análisis sistemático de obras con técnicas no destructivas (luz UV, resonancia térmica)
- Búsqueda de símbo